### 1) Imports + load + sample 500k

In [3]:
import sys
sys.path.insert(0, "..")

In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split

from src.config import RANDOM_SEED, TEST_SIZE, TOP_K
from src.metrics import mapk, hit_rate_at_k
from src.model_utils import topk_from_proba

# IMPORTANT: use fe_v1 now
from src.fe_v1 import make_features  # <-- adjust import if file name differs

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_PATH = "../data/processed/df_model.parquet"
df = pd.read_parquet(DATA_PATH)

df = df.sample(n=500_000, random_state=RANDOM_SEED).reset_index(drop=True)

X, y = make_features(df)
print(X.shape, y.nunique())


(500000, 17) 100


### 2) Train/val/test split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_SEED, stratify=y_train
)


### 3) Define cat vs numeric columns (we’ll adapt to fe_v1)

In [6]:
cat_cols = [c for c in X.columns if X[c].dtype == "object"]  # strings
# plus ID-like ints (common in Expedia)
id_like = [
    "site_name","posa_continent","user_location_country","user_location_region",
    "srch_destination_id","srch_destination_type_id","channel"
]
cat_cols += [c for c in id_like if c in X.columns]
cat_cols = sorted(set(cat_cols))

num_cols = [c for c in X.columns if c not in cat_cols]

print("cat:", len(cat_cols), "num:", len(num_cols))


cat: 9 num: 8


### Column groups

In [8]:
cat_int_cols = [
    "site_name",
    "posa_continent",
    "user_location_country",
    "user_location_region",
    "srch_destination_id",
    "srch_destination_type_id",
    "channel",
]
cat_str_cols = ["stay_type", "distance_bucket"]

num_cols = ["srch_adults_cnt", "srch_children_cnt", "srch_rm_cnt", "checkin_month", "length_of_stay"]
bin_cols = ["is_mobile", "is_package", "distance_missing"]


### TF preprocessing (lookup+embedding + normalization)

In [9]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers

inputs = {}

# int categorical inputs
for c in cat_int_cols:
    inputs[c] = tf.keras.Input(shape=(1,), name=c, dtype=tf.int64)

# string categorical inputs
for c in cat_str_cols:
    inputs[c] = tf.keras.Input(shape=(1,), name=c, dtype=tf.string)

# numeric + binary as float
for c in num_cols + bin_cols:
    inputs[c] = tf.keras.Input(shape=(1,), name=c, dtype=tf.float32)

encoded = []

def emb_dim_for_vocab(vocab_size: int) -> int:
    # simple, safe heuristic for CPU
    return int(min(32, round(np.sqrt(vocab_size) + 1)))

# int categorical: IntegerLookup -> Embedding -> Flatten
for c in cat_int_cols:
    lookup = layers.IntegerLookup(output_mode="int", name=f"{c}_lookup")
    lookup.adapt(X_train[c].values)  # adapt on train only
    v = lookup.vocabulary_size()
    d = emb_dim_for_vocab(v)

    x = lookup(inputs[c])
    x = layers.Embedding(v, d, name=f"{c}_emb")(x)
    x = layers.Reshape((d,), name=f"{c}_flat")(x)
    encoded.append(x)

# string categorical: StringLookup -> Embedding -> Flatten
for c in cat_str_cols:
    lookup = layers.StringLookup(output_mode="int", name=f"{c}_lookup")
    lookup.adapt(X_train[c].astype(str).values)
    v = lookup.vocabulary_size()
    d = emb_dim_for_vocab(v)

    x = lookup(inputs[c])
    x = layers.Embedding(v, d, name=f"{c}_emb")(x)
    x = layers.Reshape((d,), name=f"{c}_flat")(x)
    encoded.append(x)

# numeric normalization (num + bin together)
num_stack = layers.Concatenate(name="num_concat")([inputs[c] for c in num_cols + bin_cols])

normalizer = layers.Normalization(name="num_norm")
normalizer.adapt(
    np.column_stack([X_train[c].astype("float32").values for c in num_cols + bin_cols])
)
num_normed = normalizer(num_stack)

all_features = layers.Concatenate(name="all_features")(encoded + [num_normed])


### Model v2 (L2 + tuned dropout + better LR)

In [10]:
x = layers.Dense(
    256, activation="relu",
    kernel_regularizer=regularizers.l2(1e-5)
)(all_features)
x = layers.Dropout(0.25)(x)

x = layers.Dense(
    128, activation="relu",
    kernel_regularizer=regularizers.l2(1e-5)
)(x)
x = layers.Dropout(0.15)(x)

outputs = layers.Dense(y.nunique(), activation="softmax", name="hotel_cluster")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
    loss="sparse_categorical_crossentropy",
)
model.summary()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 site_name (InputLayer)      [(None, 1)]                  0         []                            
                                                                                                  
 posa_continent (InputLayer  [(None, 1)]                  0         []                            
 )                                                                                                
                                                                                                  
 user_location_country (Inp  [(None, 1)]                  0         []                            
 utLayer)                                                                                         
                                                                                              

### Data to dict helper (matches your column types)

In [11]:
def df_to_model_input(Xdf):
    out = {}
    for c in cat_int_cols:
        out[c] = Xdf[c].values
    for c in cat_str_cols:
        out[c] = Xdf[c].astype(str).values
    for c in num_cols + bin_cols:
        out[c] = Xdf[c].astype("float32").values
    return out

train_in = df_to_model_input(X_train)
val_in   = df_to_model_input(X_val)
test_in  = df_to_model_input(X_test)


### Training (CPU-friendly)

In [12]:
ckpt_path = "notebooks/checkpoints/dnn_v2_best.weights.h5"
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True, save_weights_only=True),
]

history = model.fit(
    train_in, y_train.values,
    validation_data=(val_in, y_val.values),
    epochs=20,
    batch_size=4096,   # big batch = faster on CPU
    callbacks=callbacks,
    verbose=1,
)


Epoch 1/20
74/74 [==============================] - 4s 30ms/step - loss: 4.4532 - val_loss: 4.3042
Epoch 2/20
74/74 [==============================] - 2s 28ms/step - loss: 4.2217 - val_loss: 4.0251
Epoch 3/20
74/74 [==============================] - 2s 25ms/step - loss: 3.8876 - val_loss: 3.7259
Epoch 4/20
74/74 [==============================] - 2s 28ms/step - loss: 3.6978 - val_loss: 3.6128
Epoch 5/20
74/74 [==============================] - 2s 27ms/step - loss: 3.5868 - val_loss: 3.5306
Epoch 6/20
74/74 [==============================] - 2s 26ms/step - loss: 3.5027 - val_loss: 3.4718
Epoch 7/20
74/74 [==============================] - 2s 27ms/step - loss: 3.4351 - val_loss: 3.4274
Epoch 8/20
74/74 [==============================] - 2s 25ms/step - loss: 3.3768 - val_loss: 3.3912
Epoch 9/20
74/74 [==============================] - 2s 25ms/step - loss: 3.3270 - val_loss: 3.3644
Epoch 10/20
74/74 [==============================] - 2s 25ms/step - loss: 3.2873 - val_loss: 3.3438
Epoch 11/

### Evaluation (MAP@5 + Hit@5)

In [13]:
proba = model.predict(test_in, batch_size=8192)

classes = np.sort(y_train.unique())
topk = topk_from_proba(proba, classes=classes, k=TOP_K)

y_true = y_test.values.tolist()
print("MAP@5:", mapk(y_true, topk, k=TOP_K))
print("Hit@5:", hit_rate_at_k(y_true, topk, k=TOP_K))


16/16 [==============================] - 0s 9ms/step
MAP@5: 0.2843342666666666
Hit@5: 0.478328


In [15]:
def emb_dim_for_col(col: str, vocab_size: int) -> int:
    if col == "srch_destination_id":
        return 64
    return int(min(32, round(np.sqrt(vocab_size) + 1)))


In [16]:
d = emb_dim_for_vocab(v)


In [17]:
d = emb_dim_for_col(c, v)


In [18]:
x = layers.Dense(256, activation="relu")(all_features)
x = layers.Dropout(0.15)(x)

x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.10)(x)

outputs = layers.Dense(y.nunique(), activation="softmax")(x)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
)


In [19]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True, save_weights_only=True),
]

history = model.fit(
    train_in, y_train.values,
    validation_data=(val_in, y_val.values),
    epochs=50,
    batch_size=4096,
    callbacks=callbacks,
    verbose=1,
)


Epoch 1/50
74/74 [==============================] - 3s 31ms/step - loss: 3.0464 - val_loss: 3.2455
Epoch 2/50
74/74 [==============================] - 2s 28ms/step - loss: 3.0182 - val_loss: 3.2429
Epoch 3/50
74/74 [==============================] - 2s 27ms/step - loss: 2.9984 - val_loss: 3.2429
Epoch 4/50
74/74 [==============================] - 2s 25ms/step - loss: 2.9822 - val_loss: 3.2395
Epoch 5/50
74/74 [==============================] - 2s 24ms/step - loss: 2.9664 - val_loss: 3.2393
Epoch 6/50
74/74 [==============================] - 2s 24ms/step - loss: 2.9553 - val_loss: 3.2408
Epoch 7/50
74/74 [==============================] - 2s 24ms/step - loss: 2.9432 - val_loss: 3.2382
Epoch 8/50
74/74 [==============================] - 2s 24ms/step - loss: 2.9326 - val_loss: 3.2396
Epoch 9/50
74/74 [==============================] - 2s 23ms/step - loss: 2.9207 - val_loss: 3.2415
Epoch 10/50
74/74 [==============================] - 2s 24ms/step - loss: 2.9129 - val_loss: 3.2405


In [21]:
proba = model.predict(test_in, batch_size=8192)

classes = np.sort(y_train.unique())
topk = topk_from_proba(proba, classes=classes, k=TOP_K)

y_true = y_test.values.tolist()
print("MAP@5:", mapk(y_true, topk, k=TOP_K))
print("Hit@5:", hit_rate_at_k(y_true, topk, k=TOP_K))


16/16 [==============================] - 0s 9ms/step
MAP@5: 0.2925272
Hit@5: 0.49336
